In [1]:
from huggingface_hub import login
login()

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="google/gemma-4-E4B",
    local_dir="./models/gemma-4-E4B",
)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

'/home/aliozkaya/uni/467/term_project/src/models/gemma-4-E4B'

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import pandas as pd

In [4]:
train_df = pd.read_csv("./data/train.csv")

def format_example(row):
    return f"Write song lyrics.\n\n{row['clean']}"

train_df["text"] = train_df.apply(format_example, axis=1)

In [5]:
model_path = "./models/gemma-4-E4B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

model = prepare_model_for_kbit_training(model)

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

/home/aliozkaya/uni/467/term_project/src/.venv/lib/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [6]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=r"model\.language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)",
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    use_dora=True
)

In [7]:
for name, _ in model.named_modules():
    if "language_model" in name and "q_proj" in name and "layers.0" in name:
        print(repr(name))


'model.language_model.layers.0.self_attn.q_proj'


In [8]:
from datasets import Dataset
from trl import SFTConfig

artist = "Tool"
artist_df = train_df[train_df["artist"] == artist].reset_index(drop=True)
artist_dataset = Dataset.from_pandas(artist_df[["text"]])

model_fresh = get_peft_model(model, lora_config)

output_dir = f"./adapters/{artist.lower().replace(' ', '_')}"

training_args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    max_length=512,
    bf16=True,
    logging_steps=5,
    save_strategy="epoch",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    report_to="none",
    weight_decay=0.05
)

trainer = SFTTrainer(
    model=model_fresh,
    train_dataset=artist_dataset,
    args=training_args,
    processing_class=tokenizer,
)

trainer.train()
model_fresh.save_pretrained(output_dir)
print(f"Adapter saved to {output_dir}")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/67 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/67 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 1}.


Step,Training Loss
5,3.027813
10,2.227389
15,1.931940
20,1.881972
25,2.087544
30,1.849209
35,1.692912
40,1.622220
45,1.574594
50,1.240158


Adapter saved to ./adapters/tool


In [9]:
for name, param in model_fresh.named_parameters():
    if "lora" in name and param.requires_grad:
        print(name, param.data.abs().mean().item())

base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_A.default.weight 0.0098876953125
base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_B.default.weight 0.000766754150390625
base_model.model.model.language_model.layers.0.self_attn.q_proj.lora_magnitude_vector.default.weight 1.0078125
base_model.model.model.language_model.layers.0.self_attn.k_proj.lora_A.default.weight 0.00994873046875
base_model.model.model.language_model.layers.0.self_attn.k_proj.lora_B.default.weight 0.00079345703125
base_model.model.model.language_model.layers.0.self_attn.k_proj.lora_magnitude_vector.default.weight 1.015625
base_model.model.model.language_model.layers.0.self_attn.v_proj.lora_A.default.weight 0.010009765625
base_model.model.model.language_model.layers.0.self_attn.v_proj.lora_B.default.weight 0.000820159912109375
base_model.model.model.language_model.layers.0.self_attn.v_proj.lora_magnitude_vector.default.weight 1.0234375
base_model.model.model.language_model.layers.

In [10]:
for name, module in model.named_modules():
    if "layers.0." in name and any(k in name for k in ["q_proj", "gate_proj"]):
        if "vision" not in name and "audio" not in name:
            print(name, type(module))

model.language_model.layers.0.self_attn.q_proj <class 'peft.tuners.lora.bnb.Linear4bit'>
model.language_model.layers.0.self_attn.q_proj.base_layer <class 'bitsandbytes.nn.modules.Linear4bit'>
model.language_model.layers.0.self_attn.q_proj.lora_dropout <class 'torch.nn.modules.container.ModuleDict'>
model.language_model.layers.0.self_attn.q_proj.lora_dropout.default <class 'torch.nn.modules.dropout.Dropout'>
model.language_model.layers.0.self_attn.q_proj.lora_A <class 'torch.nn.modules.container.ModuleDict'>
model.language_model.layers.0.self_attn.q_proj.lora_A.default <class 'torch.nn.modules.linear.Linear'>
model.language_model.layers.0.self_attn.q_proj.lora_B <class 'torch.nn.modules.container.ModuleDict'>
model.language_model.layers.0.self_attn.q_proj.lora_B.default <class 'torch.nn.modules.linear.Linear'>
model.language_model.layers.0.self_attn.q_proj.lora_embedding_A <class 'torch.nn.modules.container.ParameterDict'>
model.language_model.layers.0.self_attn.q_proj.lora_embedding_B 

In [11]:
import peft, transformers
print(peft.__version__, transformers.__version__)

0.19.1 5.8.1


In [12]:
from trl import SFTConfig
help(SFTConfig.__init__)

Help on function __init__ in module trl.trainer.sft_config:

__init__(
    self,
    output_dir: str | None = None,
    per_device_train_batch_size: int = 8,
    num_train_epochs: float = 3.0,
    max_steps: int = -1,
    learning_rate: float = 2e-05,
    lr_scheduler_type: SchedulerType | str = 'linear',
    lr_scheduler_kwargs: dict | str | None = None,
    warmup_steps: float = 0,
    optim: OptimizerNames | str = 'adamw_torch_fused',
    optim_args: str | None = None,
    weight_decay: float = 0.0,
    adam_beta1: float = 0.9,
    adam_beta2: float = 0.999,
    adam_epsilon: float = 1e-08,
    optim_target_modules: None | str | list[str] = None,
    gradient_accumulation_steps: int = 1,
    average_tokens_across_devices: bool = True,
    max_grad_norm: float = 1.0,
    label_smoothing_factor: float = 0.0,
    bf16: bool | None = None,
    fp16: bool = False,
    bf16_full_eval: bool = False,
    fp16_full_eval: bool = False,
    tf32: bool | None = None,
    gradient_checkpointing: